Prathmesh Nitnaware


In [33]:
!pip install pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [34]:
import pandas as pd 


In [35]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [36]:
train.head()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


In [37]:
train.describe()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
count,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000
mean,314999.500000,54.136706,0.714735,3.312752,130.497433,245.011814,0.079987,0.981660,152.816763,0.273725,0.716028,1.455871,0.451040,4.618873
std,181865.479132,8.256301,0.451541,0.851615,14.975802,33.681581,0.271274,0.998783,19.112927,0.445870,0.948472,0.545192,0.798549,1.950007
min,0.000000,29.000000,0.000000,1.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,1.000000,0.000000,3.000000
25%,157499.750000,48.000000,0.000000,3.000000,120.000000,223.000000,0.000000,0.000000,142.000000,0.000000,0.000000,1.000000,0.000000,3.000000
50%,314999.500000,54.000000,1.000000,4.000000,130.000000,243.000000,0.000000,0.000000,157.000000,0.000000,0.100000,1.000000,0.000000,3.000000
75%,472499.250000,60.000000,1.000000,4.000000,140.000000,269.000000,0.000000,2.000000,166.000000,1.000000,1.400000,2.000000,1.000000,7.000000
max,629999.000000,77.000000,1.000000,4.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,3.000000,3.000000,7.000000


In [38]:
train.isnull().sum()



id                         0
Age                        0
Sex                        0
Chest pain type            0
BP                         0
Cholesterol                0
FBS over 120               0
EKG results                0
Max HR                     0
Exercise angina            0
ST depression              0
Slope of ST                0
Number of vessels fluro    0
Thallium                   0
Heart Disease              0
dtype: int64

In [39]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

cat_cols = train.select_dtypes(include='object').columns

preprocess = ColumnTransformer([
    ('one_hot', OneHotEncoder(), cat_cols)
], remainder='passthrough')



In [40]:
train_clean = train.dropna(subset=["Heart Disease"])
X = train_clean.drop(["id", "Heart Disease"], axis=1)
y = train_clean["Heart Disease"]
print("NaNs in X:", X.isna().sum().sum())
print("NaNs in y:", y.isna().sum())


NaNs in X: 0
NaNs in y: 0


In [41]:
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", x_train.shape)
print("Validation shape:", x_val.shape)


Train shape: (504000, 13)
Validation shape: (126000, 13)


In [42]:
print(train.columns)

Index(['id', 'Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol',
       'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina',
       'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium',
       'Heart Disease'],
      dtype='object')


In [43]:
print("NaNs in X:", X.isna().sum().sum())
print("NaNs in y:", y.isna().sum())


NaNs in X: 0
NaNs in y: 0


In [44]:
print(y.value_counts())

Heart Disease
Absence     347546
Presence    282454
Name: count, dtype: int64


In [45]:
y = train_clean["Heart Disease"].map({
    "Absence": 0,
    "Presence": 1
})

x=train_clean.drop(["id", "Heart Disease"], axis=1)

In [46]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

In [47]:
from sklearn.linear_model import LogisticRegression

log_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

log_model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [48]:
from sklearn.metrics import roc_auc_score

preds = log_model.predict_proba(x_val)[:,1]
print("Logistic AUC:", roc_auc_score(y_val, preds))

Logistic AUC: 0.951546958953206


In [49]:
import pandas as pd

temp = X.copy()
temp["target"] = y

corr = temp.corr(numeric_only=True)["target"].sort_values(ascending=False)
print(corr)


target                     1.000000
Thallium                   0.605776
Chest pain type            0.460684
Exercise angina            0.441864
Number of vessels fluro    0.438604
ST depression              0.430641
Slope of ST                0.415050
Sex                        0.342446
EKG results                0.218961
Age                        0.212091
Cholesterol                0.082753
FBS over 120               0.033570
BP                        -0.005181
Max HR                    -0.440985
Name: target, dtype: float64


In [50]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    log_model,
    X,
    y,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

print("Fold AUCs:", scores)
print("Mean CV AUC:", scores.mean())


Fold AUCs: [0.95075056 0.94996061 0.95078322 0.95021808 0.95073789]
Mean CV AUC: 0.9504900722631844


In [51]:
from lightgbm import LGBMClassifier

lgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.02,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    )
])

lgb_model.fit(X, y)

scores = cross_val_score(lgb_model, x, y, cv=cv, scoring="roc_auc", n_jobs=-1)

print("LightGBM CV AUC:", scores.mean())

[LightGBM] [Info] Number of positive: 282454, number of negative: 347546
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031259 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 423
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448340 -> initscore=-0.207381
[LightGBM] [Info] Start training from score -0.207381
LightGBM CV AUC: 0.9552747552340728


In [52]:
cat_features = X.select_dtypes(include=['object']).columns.tolist()
print("Categorical features:", cat_features)


Categorical features: []


In [53]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=800,
    learning_rate=0.05,
    depth=6,
    eval_metric='AUC',
    random_state=42,
    verbose=100,
    early_stopping_rounds=100
)

cat_model.fit(x_train, y_train, eval_set=(x_val, y_val))


0:	test: 0.9398405	best: 0.9398405 (0)	total: 83ms	remaining: 1m 6s
100:	test: 0.9547466	best: 0.9547466 (100)	total: 8.15s	remaining: 56.4s
200:	test: 0.9552934	best: 0.9552934 (200)	total: 15.6s	remaining: 46.6s
300:	test: 0.9555768	best: 0.9555768 (300)	total: 22.4s	remaining: 37.2s
400:	test: 0.9558930	best: 0.9558930 (400)	total: 29.3s	remaining: 29.2s
500:	test: 0.9560598	best: 0.9560598 (500)	total: 35.9s	remaining: 21.4s
600:	test: 0.9561347	best: 0.9561347 (600)	total: 42s	remaining: 13.9s
700:	test: 0.9561855	best: 0.9561855 (700)	total: 48.6s	remaining: 6.86s
799:	test: 0.9562252	best: 0.9562252 (799)	total: 54.3s	remaining: 0us

bestTest = 0.9562251852
bestIteration = 799



CatBoostClassifier(depth=6, early_stopping_rounds=100, eval_metric='AUC', iterations=800, learning_rate=0.05, random_state=42, verbose=100)

In [54]:
from sklearn.metrics import roc_auc_score

cat_preds = cat_model.predict_proba(x_val)[:,1]
cat_auc = roc_auc_score(y_val, cat_preds)
print("CatBoost Validation AUC:", cat_auc)


CatBoost Validation AUC: 0.9562251851533464


In [55]:
x_full = train.drop(["id", "Heart Disease"], axis=1)
y_full = train["Heart Disease"]


In [56]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
import pandas as pd

train = train.dropna(subset=["Heart Disease"])

x = train.drop(["id", "Heart Disease"], axis=1)
y = train["Heart Disease"]
test_x = test.drop("id", axis=1)

num_imputer = SimpleImputer(strategy="median")
x_full = pd.DataFrame(num_imputer.fit_transform(x), columns=x.columns)
test_full = pd.DataFrame(num_imputer.transform(test_x), columns=test_x.columns)

x_train, x_val, y_train, y_val = train_test_split(x_full, y, test_size=0.2, stratify=y, random_state=42)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(x_train, y_train)
log_val_preds = log_model.predict_proba(x_val)[:, 1]
print("Logistic AUC:", roc_auc_score(y_val, log_val_preds))

lgb_model = LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8)
lgb_model.fit(x_train, y_train)
lgb_val_preds = lgb_model.predict_proba(x_val)[:, 1]
print("LightGBM AUC:", roc_auc_score(y_val, lgb_val_preds))

cat_model = CatBoostClassifier(iterations=800, learning_rate=0.05, depth=6, eval_metric='AUC', random_state=42, verbose=0, early_stopping_rounds=100)
cat_model.fit(x_train, y_train, eval_set=(x_val, y_val))
cat_val_preds = cat_model.predict_proba(x_val)[:, 1]
print("CatBoost AUC:", roc_auc_score(y_val, cat_val_preds))

log_test_preds = log_model.predict_proba(test_full)[:, 1]
lgb_test_preds = lgb_model.predict_proba(test_full)[:, 1]
cat_test_preds = cat_model.predict_proba(test_full)[:, 1]

final_preds = (log_test_preds + lgb_test_preds + cat_test_preds) / 3
submission = pd.DataFrame({"id": test["id"], "Heart Disease": final_preds})
submission.to_csv("submission.csv", index=False)

c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic AUC: 0.9515458098771161
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.053252 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 416
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warn